# 03 - Configuring the agent: tools, permissions, budgets, prompts

Notebook 02 got an agent running. This one is about the four settings that
decide whether it is safe to deploy.

- **What can it reach?** (tool availability)
- **What runs without asking?** (permissions)
- **When does it stop?** (turn and budget caps)
- **How does it behave?** (system prompt)

**Docs:** [Permissions](https://code.claude.com/docs/en/agent-sdk/permissions) ·
[Agent loop: control how the loop runs](https://code.claude.com/docs/en/agent-sdk/agent-loop#control-how-the-loop-runs)

In [ ]:
# --- boot cell: run this first in every notebook -------------------------
import os, sys, json, asyncio
from pathlib import Path
from dotenv import load_dotenv

# The SDK reads ANTHROPIC_API_KEY from the process environment.
# It does NOT parse .env for you, so we do it here.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY is not set. Copy .env.example to .env and add your key."
)

MODEL = os.environ.get("COURSE_MODEL") or None   # None = SDK default
print("key loaded, model =", MODEL or "(SDK default)")

In [ ]:
from claude_agent_sdk import (
    query, ClaudeAgentOptions, AssistantMessage, ResultMessage,
    TextBlock, ToolUseBlock,
)

SCRATCH = ROOT / "workspace" / "_scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)
(SCRATCH / "notes.txt").write_text("alpha\nbeta\ngamma\n", encoding="utf-8")

async def run(prompt, **opts):
    """Run one query and print a compact trace. Used all through this notebook."""
    opts.setdefault("cwd", str(SCRATCH))
    opts.setdefault("setting_sources", [])
    opts.setdefault("max_turns", 8)
    opts.setdefault("model", MODEL)
    final = None
    try:
        async for m in query(prompt=prompt, options=ClaudeAgentOptions(**opts)):
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, ToolUseBlock):
                        print(f"   TOOL {b.name} {str(b.input)[:70]}")
                    elif isinstance(b, TextBlock) and b.text.strip():
                        print(f"   ...  {b.text.strip()[:100]}")
            elif isinstance(m, ResultMessage):
                final = m
                print(f"   ==> {m.subtype} | {m.num_turns} turns | "
                      f"${m.total_cost_usd or 0:.4f}")
                if m.permission_denials:
                    for d in m.permission_denials:
                        print(f"   DENIED: {d}")
    except Exception as exc:
        print(f"   raised: {type(exc).__name__}: {str(exc)[:90]}")
    return final

print("helper ready")

## 3.1 The built-in tools

| Category | Tools | What they do |
|---|---|---|
| Files | `Read`, `Write`, `Edit` | read, create, modify files |
| Search | `Glob`, `Grep` | find files by pattern, search content by regex |
| Execution | `Bash` | shell commands, scripts, git |
| Web | `WebSearch`, `WebFetch` | search the web, fetch and parse pages |
| Discovery | `ToolSearch` | load tool schemas on demand instead of upfront |
| Orchestration | `Agent`, `Skill`, `AskUserQuestion`, `TaskCreate`, `TaskUpdate` | spawn subagents, invoke skills, ask the user, track tasks |

Useful presets:

| Tools | Agent shape |
|---|---|
| `Read`, `Glob`, `Grep` | read-only analysis |
| `Read`, `Edit`, `Glob` | analyse and modify code |
| `Read`, `Edit`, `Bash`, `Glob`, `Grep` | full automation |

**Docs:** [Tools reference](https://code.claude.com/docs/en/tools-reference)

## 3.2 Two different layers, easily confused

This is the single most common source of "why did my restriction not work".

| Option | Layer | Effect |
|---|---|---|
| `tools=["Read","Grep"]` | **availability** | only these built-ins exist in Claude's context. MCP tools unaffected |
| `tools=[]` | **availability** | no built-ins at all; only your MCP tools |
| `disallowed_tools=["Bash"]` | **availability** | bare name — removes the tool from context entirely |
| `allowed_tools=["Read"]` | **permission** | auto-approved. Unlisted tools still *exist* and fall through to the permission mode |
| `disallowed_tools=["Bash(rm *)"]` | **permission** | scoped rule — `Bash` stays visible, matching calls are denied |

**Availability** is about what Claude can see. **Permission** is about what runs
without asking.

Removing a tool from availability is cheaper and safer than denying it later: if
Claude cannot see `Bash`, it never spends a turn trying it.

### A genuinely read-only agent

Availability *and* permission, both locked down.

In [ ]:
await run(
    "Summarise notes.txt in one line, then append the word 'delta' to it.",
    tools=["Read", "Glob", "Grep"],          # availability: no Write, no Edit, no Bash
    allowed_tools=["Read", "Glob", "Grep"],  # permission: these run unprompted
    permission_mode="dontAsk",               # anything else -> denied, never prompted
)
print("\nfile on disk is unchanged:")
print((SCRATCH / "notes.txt").read_text())

Claude summarised the file and reported that it could not append. It did
not fail, hang, or wait for a prompt that no one would answer — it worked with
what it had and said so. That is the behaviour you want from an unattended
agent.

## 3.3 Permission modes

| Mode | Behaviour | Use for |
|---|---|---|
| `default` | unmatched tools go to your `can_use_tool` callback; no callback means deny | interactive apps with an approval UI |
| `acceptEdits` | auto-approves file edits and common filesystem commands (`mkdir`, `touch`, `mv`, `cp`) | prototyping in an isolated directory |
| `plan` | explore and plan; file edits are never auto-approved | "propose changes, do not make them" |
| `dontAsk` | never prompts — pre-approved runs, everything else is denied | headless agents with a fixed tool surface |
| `auto` | a model classifier approves or denies | autonomous agents that still want guardrails |
| `bypassPermissions` | runs everything (with narrow exceptions) | CI and containers **only** |

Two warnings worth internalising:

- **`allowed_tools` does not constrain `bypassPermissions`.** Setting
  `allowed_tools=["Read"]` alongside `bypassPermissions` still approves `Bash`,
  `Write`, and everything else. To block a tool in that mode, use
  `disallowed_tools`.
- **Auto-approved tools never reach `can_use_tool`.** If a tool is approved by
  an allow rule or by the mode, your callback is skipped for it. For a check
  that must run on *every* call, use a `PreToolUse` hook (notebook 07).

**Docs:** [Permission modes](https://code.claude.com/docs/en/agent-sdk/permissions#permission-modes)

## 3.4 The evaluation order

When Claude requests a tool, the SDK decides in this fixed order:

```
   tool request
        │
        ▼
   1. HOOKS ──────────── deny ──▶ BLOCKED
        │
        ▼
   2. DENY rules ─────── match ─▶ BLOCKED       (wins even in bypassPermissions)
        │
        ▼
   3. ASK rules ──────── match ─▶ can_use_tool
        │
        ▼
   4. PERMISSION MODE ── approve ▶ EXECUTE      (bypassPermissions / acceptEdits)
        │
        ▼
   5. ALLOW rules ────── match ─▶ EXECUTE
        │
        ▼
   6. can_use_tool ───── decide ─▶ EXECUTE or BLOCKED
                                   (in dontAsk: skipped, denied)
```

Read it top-down and two facts fall out:

- **A hook deny beats everything**, including `bypassPermissions`. Hooks are
  the only control that applies unconditionally.
- **A deny rule beats an allow rule**, always. There is no ordering subtlety to
  get wrong.

## 3.5 Scoped rules

Rules can be narrower than a whole tool.

```python
allowed_tools=["Bash(npm run test:*)"]   # only this command family
disallowed_tools=["Bash(rm *)"]          # Bash stays available; rm calls denied
disallowed_tools=["Edit(//etc/**)"]      # absolute path: // prefix
disallowed_tools=["mcp__*"]              # every MCP tool from every server
allowed_tools=["mcp__github__get_*"]     # a server's get_ tools
```

Three details that bite:

- `Edit(path)` governs **all** file-writing built-ins, `Write` and
  `NotebookEdit` included. A `Write(path)` rule is never matched.
- `//path` is an absolute filesystem path. A single leading slash anchors at the
  rule's source — for rules passed in code, that is the working directory.
- Allow rules accept globs only after a literal `mcp__<server>__` prefix.
  `allowed_tools=["*"]` is ignored with a startup warning.

In [ ]:
await run(
    "Run `echo hello`, then run `rm notes.txt`.",
    tools=["Bash"],
    allowed_tools=["Bash"],               # Bash is broadly approved...
    disallowed_tools=["Bash(rm *)"],      # ...but rm is denied at step 2
    permission_mode="dontAsk",
)
print("\nnotes.txt still exists:", (SCRATCH / "notes.txt").exists())

## 3.6 Stopping: turns, budget, effort

| Option | Caps | On hit |
|---|---|---|
| `max_turns` | tool-use round trips | `error_max_turns` |
| `max_budget_usd` | estimated spend, **including subagents** | `error_max_budget_usd` |
| `effort` | reasoning depth per turn: `low`/`medium`/`high`/`xhigh`/`max` | n/a — a cost lever, not a stop |

Setting a budget is a good default for anything running unattended. It is also
the only cap that bounds a subagent tree: once spend reaches the cap, spawning
another subagent fails and running background subagents are stopped.

`effort` is the cheapest quality/cost dial you have. An agent that only lists
files does not need `high`.

In [ ]:
r = await run(
    "Count the lines in notes.txt.",
    tools=["Read", "Bash"], allowed_tools=["Read", "Bash"],
    permission_mode="dontAsk",
    effort="low",              # minimal reasoning: this is a lookup, not a refactor
    max_budget_usd=0.05,       # hard ceiling
    max_turns=5,
)

## 3.7 System prompts

Three forms, and the default is not what people expect.

```python
# 1. Your own prompt. Claude Code's own instructions are NOT included.
system_prompt="You are a support triage agent. ..."

# 2. Claude Code's preset, plus your additions.
system_prompt={"type": "preset", "preset": "claude_code",
               "append": "Always follow PEP 8."}

# 3. From a file.
system_prompt={"type": "file", "path": "/path/to/prompt.md"}

# 4. Omitted entirely -> the SDK's default (no Claude Code preset)
```

**If you pass a plain string you are replacing, not extending.** That is usually
right for a purpose-built agent — you want a support agent, not a coding
assistant that also does support. Use the preset when you want the coding
behaviour and are adding to it.

**Docs:** [Modifying system prompts](https://code.claude.com/docs/en/agent-sdk/modifying-system-prompts)

In [ ]:
await run(
    "What are you for? One sentence.",
    tools=[],
    system_prompt=(
        "You are a support triage agent for a B2B SaaS company. You classify "
        "tickets, route them, and draft replies. You never write code."
    ),
    max_turns=1,
)

## 3.8 `setting_sources`: where else configuration comes from

By default `query()` loads filesystem configuration — `CLAUDE.md`, project
skills, `.claude/settings.json` permission rules, filesystem hooks — from the
`project` source.

That is convenient locally and a reproducibility hazard in production: the same
code behaves differently depending on which directory it runs in.

```python
setting_sources=[]                      # load nothing. Explicit and portable.
setting_sources=["project"]             # load ./.claude/ and ./CLAUDE.md
setting_sources=["user", "project"]     # also ~/.claude/
```

Every example in this course sets `setting_sources=[]`, so behaviour comes from
the code and nothing else. Turn it on deliberately, when you want the project's
conventions to apply.

**Docs:** [Claude Code features in the SDK](https://code.claude.com/docs/en/agent-sdk/claude-code-features)

## 3.9 A production-shaped configuration

Everything above, together. This is roughly the shape of the agent we build in
notebook 10.

```python
ClaudeAgentOptions(
    system_prompt=MY_PROMPT,          # purpose-built, not the coding preset
    mcp_servers={"support": server},  # your tools (notebook 05)
    tools=[],                         # availability: no built-ins at all
    allowed_tools=[                   # permission: exactly these, unprompted
        "mcp__support__search_kb",
        "mcp__support__save_triage",
    ],
    permission_mode="dontAsk",        # never prompt; deny the rest
    max_turns=20,                     # bound the loop
    max_budget_usd=0.50,              # bound the bill
    effort="medium",
    setting_sources=[],               # no ambient config
)
```

Read it as five sentences: this is who you are, these are your tools, nothing
else exists, nothing else runs, and here is when you stop.

## 3.10 Exercises

1. Set `permission_mode="plan"` and ask the agent to rewrite `notes.txt`. What
   does it produce instead?
2. Set `max_budget_usd=0.001` and give it real work. Confirm the subtype is
   `error_max_budget_usd`, and check whether `total_cost_usd` exceeded the cap.
3. Compare `effort="low"` and `effort="high"` on the same non-trivial prompt.
   Record turns and cost for each. Was the extra spend worth it?
4. Try `tools=["Read"]` with `allowed_tools=["Read","Write"]` and ask it to
   write a file. Which layer wins, and why?

In [ ]:
# Scratch space for the exercises.

## Next

[**04 - User interaction**](04-user-interaction-askuserquestion.ipynb) — let the
agent pause mid-task and ask you a question with `AskUserQuestion`.

---

### References

- [Configure permissions](https://code.claude.com/docs/en/agent-sdk/permissions)
- [Permission modes](https://code.claude.com/docs/en/permission-modes)
- [Control how the loop runs](https://code.claude.com/docs/en/agent-sdk/agent-loop#control-how-the-loop-runs)
- [Modifying system prompts](https://code.claude.com/docs/en/agent-sdk/modifying-system-prompts)
- [Settings reference](https://code.claude.com/docs/en/settings-reference#permission-settings)